# NumPy 행렬 계산 튜토리얼

이 노트북은 로보틱스, 시뮬레이션, 좌표변환을 다룰 때 자주 쓰는 NumPy 행렬 계산을 예제로 익히는 자료입니다.

## 목표

1. 벡터, 행렬, shape를 정확히 구분한다.
2. 원소별 연산과 행렬곱의 차이를 이해한다.
3. 선형시스템, 역행렬, rank, least squares를 계산한다.
4. 고유값, SVD의 기본 사용법을 익힌다.
5. 로보틱스에서 쓰는 회전행렬과 동차변환행렬을 NumPy로 계산한다.


## 0. 준비

`np.set_printoptions()`는 출력이 너무 길거나 지저분하게 보이지 않도록 설정합니다.

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

작은 helper를 하나 만들어두면 shape, dtype 확인이 편합니다.

In [ ]:
def describe(name, x):
    x = np.asarray(x)
    print(f"{name}: shape={x.shape}, ndim={x.ndim}, dtype={x.dtype}")
    print(x)
    print()

## 1. 벡터와 행렬의 shape

NumPy에서 `[1, 2, 3]`은 1차원 배열입니다. 수학에서 말하는 행벡터 `(1 x 3)` 또는 열벡터 `(3 x 1)`와 shape가 다릅니다.

In [ ]:
v = np.array([1, 2, 3])
row = np.array([[1, 2, 3]])
col = np.array([[1], [2], [3]])

describe("v", v)
describe("row", row)
describe("col", col)

행벡터/열벡터가 명확해야 행렬곱 결과도 예측하기 쉽습니다.

In [ ]:
print("row @ col =")
print(row @ col)

print("col @ row =")
print(col @ row)

## 2. 행렬 만들기

자주 쓰는 생성 함수입니다.

- `np.array`: 직접 입력
- `np.zeros`, `np.ones`: 0 또는 1로 채우기
- `np.eye`: 단위행렬
- `np.diag`: 대각행렬 또는 대각성분 추출
- `np.arange(...).reshape(...)`: 연속값으로 만든 뒤 행렬 모양으로 변환

In [ ]:
A = np.array([[1, 2, 3], [4, 5, 6]], dtype=float)
Z = np.zeros((2, 3))
O = np.ones((2, 3))
I = np.eye(3)
D = np.diag([10, 20, 30])
R = np.arange(1, 10).reshape(3, 3)

describe("A", A)
describe("Z", Z)
describe("O", O)
describe("I", I)
describe("D", D)
describe("R", R)

행렬을 붙일 때는 축(axis)을 명확히 봐야 합니다.

In [ ]:
top = np.array([[1, 2], [3, 4]])
bottom = np.array([[5, 6]])
right = np.array([[10], [20]])

vertical = np.vstack([top, bottom])
horizontal = np.hstack([top, right])

describe("vertical", vertical)
describe("horizontal", horizontal)

## 3. 원소별 연산 vs 행렬곱

NumPy에서 `*`는 원소별 곱이고, `@`는 행렬곱입니다. 로보틱스 코드에서 가장 많이 헷갈리는 부분입니다.

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[10, 20], [30, 40]])

describe("A + B", A + B)
describe("A * B  # element-wise", A * B)
describe("A @ B  # matrix multiplication", A @ B)

행렬곱은 앞 행렬의 열 개수와 뒤 행렬의 행 개수가 같아야 합니다.

In [ ]:
M = np.array([[1, 2, 3], [4, 5, 6]])      # shape: (2, 3)
N = np.array([[1, 2], [3, 4], [5, 6]])    # shape: (3, 2)

describe("M @ N", M @ N)
describe("N @ M", N @ M)

In [ ]:
try:
    bad = M @ np.array([1, 2])
except ValueError as e:
    print("shape가 맞지 않으면 ValueError가 납니다:")
    print(e)

## 4. 전치, 대각, trace, norm

기본 선형대수 연산입니다.

In [ ]:
A = np.array([[1, 2, 3], [4, 5, 6]])
S = np.array([[2, -1], [-1, 2]])

describe("A", A)
describe("A.T", A.T)

print("diag(S):", np.diag(S))
print("trace(S):", np.trace(S))
print("Frobenius norm of S:", np.linalg.norm(S))
print("2-norm of vector [3, 4]:", np.linalg.norm([3, 4]))

내적과 외적도 shape를 보면 이해가 쉽습니다.

In [ ]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])

print("inner product:", a @ b)
describe("outer product", np.outer(a, b))

## 5. Broadcasting

Broadcasting은 shape가 다른 배열을 자동으로 확장해서 계산하는 규칙입니다. 편리하지만 의도하지 않은 계산이 생길 수 있으니 shape를 자주 확인하세요.

In [ ]:
X = np.arange(12).reshape(3, 4)
row_offset = np.array([100, 200, 300, 400])      # shape: (4,)
col_scale = np.array([[1], [10], [100]])         # shape: (3, 1)

describe("X", X)
describe("X + row_offset", X + row_offset)
describe("X * col_scale", X * col_scale)

특히 `(3,)`와 `(3, 1)`은 다릅니다.

In [ ]:
v1 = np.array([1, 2, 3])
v2 = np.array([[1], [2], [3]])

describe("v1 + v1", v1 + v1)
describe("v2 + v2", v2 + v2)
describe("v1 + v2", v1 + v2)

## 6. 선형시스템 풀기: Ax = b

이론적으로는 `x = A^{-1} b`라고 쓰지만, 코드에서는 보통 `np.linalg.solve(A, b)`를 사용합니다. 더 빠르고 수치적으로 안정적입니다.

In [ ]:
A = np.array([[3, 1], [1, 2]], dtype=float)
b = np.array([9, 8], dtype=float)

x = np.linalg.solve(A, b)

print("x:", x)
print("A @ x:", A @ x)
print("residual norm:", np.linalg.norm(A @ x - b))

역행렬을 직접 구해도 같은 답이 나오지만, 실제 계산에서는 `solve`를 우선하세요.

In [ ]:
x_by_inverse = np.linalg.inv(A) @ b
print("inv(A) @ b:", x_by_inverse)
print("same result:", np.allclose(x, x_by_inverse))

## 7. determinant, rank, condition number

- `det(A)`: 행렬식
- `rank(A)`: 독립적인 행/열의 개수
- `cond(A)`: condition number. 값이 클수록 작은 오차가 결과에 크게 증폭될 수 있습니다.

In [ ]:
A = np.array([[3, 1], [1, 2]], dtype=float)
S = np.array([[1, 2], [2, 4]], dtype=float)  # 두 번째 행은 첫 번째 행의 2배

for name, matrix in [("A", A), ("S", S)]:
    print(name)
    print(matrix)
    print("det:", np.linalg.det(matrix))
    print("rank:", np.linalg.matrix_rank(matrix))
    print("cond:", np.linalg.cond(matrix))
    print()

In [ ]:
try:
    np.linalg.solve(S, np.array([1, 2], dtype=float))
except np.linalg.LinAlgError as e:
    print("singular matrix는 solve할 수 없습니다:")
    print(e)

## 8. Least Squares: 정확한 해가 없을 때

식이 너무 많거나 데이터에 노이즈가 있으면 `Ax = b`를 정확히 만족하는 `x`가 없을 수 있습니다. 이때는 `||Ax - b||`를 최소화하는 해를 구합니다.

In [ ]:
# y = c0 + c1 * t 형태의 직선을 데이터에 맞춰봅니다.
t = np.array([0, 1, 2, 3, 4], dtype=float)
y = np.array([1.1, 2.0, 2.9, 4.2, 4.8], dtype=float)

A = np.column_stack([np.ones_like(t), t])
theta, residuals, rank, singular_values = np.linalg.lstsq(A, y, rcond=None)

c0, c1 = theta
y_hat = A @ theta

print("A =")
print(A)
print("theta [c0, c1]:", theta)
print("predicted y:", y_hat)
print("residual norm:", np.linalg.norm(y_hat - y))

## 9. 고유값과 고유벡터

고유값 문제는 `A v = lambda v`를 만족하는 `lambda`, `v`를 찾는 문제입니다. 진동, 안정성, 관성행렬 분석 등에서 자주 등장합니다.

In [ ]:
A = np.array([[2, 1], [1, 2]], dtype=float)
values, vectors = np.linalg.eig(A)

print("eigenvalues:", values)
print("eigenvectors as columns:")
print(vectors)

for i, value in enumerate(values):
    v = vectors[:, i]
    print(f"check eigenpair {i}:")
    print("A @ v       =", A @ v)
    print("lambda * v  =", value * v)
    print("close:", np.allclose(A @ v, value * v))

대칭행렬은 `np.linalg.eigh`를 쓰는 것이 좋습니다. 실수 고유값과 직교 고유벡터를 더 안정적으로 계산합니다.

In [ ]:
values, vectors = np.linalg.eigh(A)
print("eigh eigenvalues:", values)
print("vectors.T @ vectors =")
print(vectors.T @ vectors)

## 10. SVD: Singular Value Decomposition

SVD는 임의의 행렬을 `A = U Sigma V^T`로 분해합니다. rank 분석, least squares, 차원 축소, 수치 안정성 체크에 매우 많이 씁니다.

In [ ]:
A = np.array([[3, 2, 2], [2, 3, -2]], dtype=float)
U, s, Vt = np.linalg.svd(A, full_matrices=False)
Sigma = np.diag(s)
A_reconstructed = U @ Sigma @ Vt

describe("U", U)
describe("singular values", s)
describe("Vt", Vt)
describe("U @ Sigma @ Vt", A_reconstructed)
print("reconstruction close:", np.allclose(A, A_reconstructed))

큰 singular value만 남기면 low-rank approximation을 만들 수 있습니다.

In [ ]:
rng = np.random.default_rng(0)
A = rng.normal(size=(6, 4))
U, s, Vt = np.linalg.svd(A, full_matrices=False)

for k in [1, 2, 3, 4]:
    A_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    error = np.linalg.norm(A - A_k)
    print(f"rank-{k} approximation error: {error:.6f}")

## 11. 2D 회전행렬

2D에서 각도 `theta`만큼 회전하는 행렬은 다음과 같습니다.

\[
R(\theta) = \begin{bmatrix}\cos\theta & -\sin\theta \\ \sin\theta & \cos\theta\end{bmatrix}
\]

회전행렬은 `R.T @ R = I`이고, 역행렬은 전치행렬입니다.

In [ ]:
def rot2(theta):
    c = np.cos(theta)
    s = np.sin(theta)
    return np.array([[c, -s], [s, c]])

theta = np.deg2rad(90)
R = rot2(theta)
p = np.array([1.0, 0.0])

print("R =")
print(R)
print("R @ p =", R @ p)
print("R.T @ R =")
print(R.T @ R)
print("inverse close to transpose:", np.allclose(np.linalg.inv(R), R.T))

## 12. 2D 동차변환행렬

로봇의 pose는 회전과 이동을 같이 갖습니다. 2D에서는 3x3 동차변환행렬로 표현할 수 있습니다.

\[
T = \begin{bmatrix} R & t \\ 0 & 1 \end{bmatrix}
\]

점 `[x, y]`는 `[x, y, 1]`로 확장해서 곱합니다.

In [ ]:
def transform2(theta, translation):
    T = np.eye(3)
    T[:2, :2] = rot2(theta)
    T[:2, 2] = np.asarray(translation, dtype=float)
    return T

def apply_transform2(T, points):
    points = np.asarray(points, dtype=float)
    points_h = np.column_stack([points, np.ones(len(points))])
    transformed_h = (T @ points_h.T).T
    return transformed_h[:, :2]

points = np.array([[0, 0], [1, 0], [1, 1], [0, 1]], dtype=float)
T_world_body = transform2(np.deg2rad(30), [0.5, 0.2])
points_world = apply_transform2(T_world_body, points)

print("T_world_body =")
print(T_world_body)
print("points in body frame:")
print(points)
print("points in world frame:")
print(points_world)

동차변환은 행렬곱으로 합성합니다. `T_world_hand = T_world_base @ T_base_hand`처럼 프레임 이름을 맞춰 쓰면 실수를 줄일 수 있습니다.

In [ ]:
T_world_base = transform2(np.deg2rad(15), [1.0, 0.0])
T_base_hand = transform2(np.deg2rad(-45), [0.4, 0.2])
T_world_hand = T_world_base @ T_base_hand

hand_origin = np.array([[0.0, 0.0]])
hand_origin_world = apply_transform2(T_world_hand, hand_origin)

print("T_world_base =")
print(T_world_base)
print("T_base_hand =")
print(T_base_hand)
print("T_world_hand =")
print(T_world_hand)
print("hand origin in world:", hand_origin_world[0])

## 13. 3D 동차변환행렬

Open3D, MuJoCo, 로봇 kinematics에서는 보통 4x4 동차변환행렬을 사용합니다.

In [ ]:
def rotz(theta):
    c = np.cos(theta)
    s = np.sin(theta)
    return np.array([
        [c, -s, 0.0],
        [s, c, 0.0],
        [0.0, 0.0, 1.0],
    ])

def make_transform3(R, p):
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = np.asarray(p, dtype=float)
    return T

T_world_tool = make_transform3(rotz(np.deg2rad(45)), [0.3, 0.2, 0.8])
point_tool = np.array([0.1, 0.0, 0.0, 1.0])
point_world = T_world_tool @ point_tool

print("T_world_tool =")
print(T_world_tool)
print("point_world =", point_world[:3])

역변환은 `np.linalg.inv(T)`로 구할 수 있습니다. 회전행렬의 성질을 이용하면 더 직접적으로 만들 수도 있습니다.

In [ ]:
T_tool_world = np.linalg.inv(T_world_tool)
recovered_tool = T_tool_world @ point_world

print("recovered point in tool frame:", recovered_tool[:3])
print("close to original:", np.allclose(recovered_tool, point_tool))

## 14. 행렬 슬라이싱

큰 행렬에서 원하는 블록을 꺼내는 방법입니다. 로보틱스에서는 4x4 pose에서 `R`, `p`를 분리할 때 자주 씁니다.

In [ ]:
T = T_world_tool
R = T[:3, :3]
p = T[:3, 3]
bottom_row = T[3, :]

describe("R", R)
describe("p", p)
describe("bottom_row", bottom_row)

## 15. 흔한 실수 모음

아래 예제들은 코드가 돌아가더라도 의미가 달라질 수 있는 부분입니다.

In [ ]:
# 실수 1: *를 행렬곱으로 착각하기
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print("A * B =")
print(A * B)
print("A @ B =")
print(A @ B)

In [ ]:
# 실수 2: 1D vector와 column vector 혼동하기
v = np.array([1, 2, 3])
col = v.reshape(-1, 1)

print("v.shape:", v.shape)
print("col.shape:", col.shape)
print("v @ v:", v @ v)
print("col.T @ col:")
print(col.T @ col)
print("col @ col.T:")
print(col @ col.T)

## 16. 연습문제

아래 셀을 직접 수정하면서 확인해보세요.

1. 3x3 단위행렬에 `[1, 2, 3]` 대각성분을 더한 행렬을 만들어보세요.
2. 2D 점 `[1, 0]`을 45도 회전하고 `[0.5, 0.2]`만큼 이동해보세요.
3. `A = [[2, 1], [1, 3]]`, `b = [1, 2]`에서 `Ax=b`를 풀고 residual을 계산하세요.
4. 임의의 5x3 행렬을 만들고 SVD로 다시 복원해보세요.

In [ ]:
# 연습문제 1
M = np.eye(3) + np.diag([1, 2, 3])
print(M)

In [ ]:
# 연습문제 2
T = transform2(np.deg2rad(45), [0.5, 0.2])
point = np.array([[1.0, 0.0]])
print(apply_transform2(T, point)[0])

In [ ]:
# 연습문제 3
A = np.array([[2, 1], [1, 3]], dtype=float)
b = np.array([1, 2], dtype=float)
x = np.linalg.solve(A, b)
print("x:", x)
print("residual:", np.linalg.norm(A @ x - b))

In [ ]:
# 연습문제 4
rng = np.random.default_rng(42)
A = rng.normal(size=(5, 3))
U, s, Vt = np.linalg.svd(A, full_matrices=False)
A_recon = U @ np.diag(s) @ Vt

print("reconstruction close:", np.allclose(A, A_recon))
print("error:", np.linalg.norm(A - A_recon))

## 빠른 요약

- 원소별 곱: `A * B`
- 행렬곱: `A @ B`
- 전치: `A.T`
- 역행렬: `np.linalg.inv(A)`
- 선형시스템: `np.linalg.solve(A, b)`
- least squares: `np.linalg.lstsq(A, b, rcond=None)`
- 고유값: `np.linalg.eig(A)` 또는 대칭행렬이면 `np.linalg.eigh(A)`
- SVD: `np.linalg.svd(A, full_matrices=False)`
- 4x4 pose에서 회전/이동 분리: `R = T[:3, :3]`, `p = T[:3, 3]`
